In [ ]:
import asyncio
import json
import os
from typing import Annotated, Any, Never

from agent_framework import (
    AgentExecutor,
    AgentExecutorRequest,
    AgentExecutorResponse,
    Message,
    WorkflowBuilder,
    WorkflowContext,
    executor,
    tool,
)
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from dotenv import load_dotenv
from IPython.display import HTML, display
from pydantic import BaseModel

print("✅ All imports successful!")


## படி 1: கட்டமைக்கப்பட்ட வெளியீடுகளுக்கான Pydantic மாடல்களை வரையறுக்கவும்

இந்த மாடல்கள் முகவர்கள் திருப்பி அளிக்கும் **ஸ்கீமா**வை வரையறுக்கின்றன. Pydantic உடன் `response_format` பயன்படுத்துதல் என்பதன் நன்மைகள்:
- ✅ வகை-பாதுகாப்பான தரவு எடுப்பு
- ✅ தானியங்கி சரிபார்ப்பு
- ✅ சுதந்திர உரை பதில்களில் வேதனைப் பிழைகள் வராதவை
- ✅ புலங்களை அடிப்படையாகக் கொண்ட சுலபமான நிபந்தனையுடைய வழி மறைவு


In [ ]:
class BookingCheckResult(BaseModel):
    """Result from checking hotel availability at a destination."""

    destination: str
    has_availability: bool
    message: str


class AlternativeResult(BaseModel):
    """Suggested alternative destination when no rooms available."""

    alternative_destination: str
    reason: str


class BookingConfirmation(BaseModel):
    """Booking suggestion when rooms are available."""

    destination: str
    action: str
    message: str


print("✅ Pydantic models defined:")
print("   - BookingCheckResult (availability check)")
print("   - AlternativeResult (alternative suggestion)")
print("   - BookingConfirmation (booking confirmation)")

## படி 2: ஹோட்டல் முன்பதிவு கருவியை உருவாக்கவும்

இந்த கருவி **availability_agent** அறைகள் கிடைக்கின்றதா என சரிபார்க்க அழைக்கும் கருவியாகும். நாம் `@ai_function` அலங்காரியை பயன்படுத்துகிறோம்:
- பைதான் செயல்பாட்டை AI-அழைக்கக்கூடிய கருவியாக மாற்ற
- LLM க்கான JSON திட்டத்தை தானாக உருவாக்க
- அளவுரு சரிபார்ப்பை கையாள
- முகவரிகள் tarafından தானாக அழைப்பதற்கான வசதியை வழங்க

இந்த டெமோவுக்காக:
- **ஸ்டோக்க்ஹோம், சீயாட்டில், டோக்கியோ, லண்டன், ஆம்ஸ்டர்டம்** → அறைகள் உள்ளன ✅
- **மற்ற எல்லா நகரங்களும்** → அறைகள் இல்லை ❌


In [ ]:
@tool(description="Check hotel room availability for a destination city")
def hotel_booking(destination: Annotated[str, "The destination city to check for hotel rooms"]) -> str:
    """
    Simulates checking hotel room availability.

    Returns JSON string with availability status.
    """
    display(
        HTML(f"""
        <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
            <strong>🔍 Tool Invoked:</strong> hotel_booking("{destination}")
        </div>
    """)
    )

    # Simulate availability check
    cities_with_rooms = ["stockholm", "seattle", "tokyo", "london", "amsterdam"]
    has_rooms = destination.lower() in cities_with_rooms

    result = {"has_availability": has_rooms, "destination": destination}

    return json.dumps(result)


print("✅ hotel_booking tool created with @tool decorator")

## படி 3: வழித்தடத்திற்கான நிபந்தனை செயல்பாடுகளை வரையறு

இத்தகவல்கள் முகவரியின் பதிலை பரிசீலித்து வேலைநடத்தை எது என்பதை தீர்மானிக்கின்றன.

**முக்கிய மாதிரி:**
1. செய்தி `AgentExecutorResponse` ஆக இருக்கிறதா என்று சரிபார்
2. கட்டமைக்கப்பட்ட வெளியீட்டை (Pydantic மாதிரி) பகுப்பாய்வு செய்
3. வழித்தடத்தைக் கட்டுப்படுத்த `True` அல்லது `False` ஐ மீட்டிடு

வேலைநடத்தை அடுத்ததாக எந்த செயற்பாட்டாளரை அழைக்க வேண்டும் என்று தீர்மானிக்க இப்பொழுதைய நிபந்தனைகளை **அடிகள்** மீது மதிப்பாய்வது. 


In [ ]:
def has_availability_condition(message: Any) -> bool:
    """
    Condition for routing when hotels ARE available.
    
    Returns True if the destination has hotel rooms.
    """
    if not isinstance(message, AgentExecutorResponse):
        return True  # Default to True if unexpected type

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)

        display(
            HTML(f"""
            <div style='padding: 12px; background: #c8e6c9; border-left: 4px solid #4caf50; border-radius: 4px; margin: 10px 0;'>
                <strong>✅ Condition Check:</strong> has_availability = <strong>{result.has_availability}</strong> for {result.destination}
            </div>
        """)
        )

        return result.has_availability
    except Exception as e:
        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffcdd2; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'>
                <strong>⚠️  Error:</strong> {str(e)}
            </div>
        """)
        )
        return False


def no_availability_condition(message: Any) -> bool:
    """
    Condition for routing when hotels are NOT available.
    
    Returns True if the destination has no hotel rooms.
    """
    if not isinstance(message, AgentExecutorResponse):
        return False

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)

        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffecb3; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
                <strong>❌ Condition Check:</strong> no_availability for {result.destination}
            </div>
        """)
        )

        return not result.has_availability
    except Exception as e:
        return False


print("✅ Condition functions defined:")
print("   - has_availability_condition (routes when rooms exist)")
print("   - no_availability_condition (routes when no rooms)")

## படி 4: தனிப்பயன் காட்சிப்படுத்தும் நிறைவேற்றுபவரை உருவாக்கவும்

நிறைவேற்றுபவர்கள் மாற்றங்களை அல்லது பக்க விளைவுகளை செயல்படுத்தும் பணிச்சூழலுக்காக கூறுகளாக இருக்கின்றன. இறுதி முடிவை காட்சிப்படுத்த தனிப்பயன் நிறைவேற்றுபவரை உருவாக்க சிலர் `@executor` அலங்காரியை பயன்படுத்துகிறோம்.

**முக்கியக் கருத்துகள்:**
- `@executor(id="...")` - ஒரு செயல்பாட்டை பணிச்சுழற்சி நிறைவேற்றுபவராக பதிவு செய்கிறது
- `WorkflowContext[Never, str]` - உள்ளீடு/வெளியேற்றத்திற்கான வகை குறிப்பு
- `ctx.yield_output(...)` - இறுதி பணிச்சுழற்சி முடிவை வழங்குகிறது


In [ ]:
@executor(id="display_result")
async def display_result(response: AgentExecutorResponse, ctx: WorkflowContext[Never, str]) -> None:
    """
    Display the final result as workflow output.
    
    This executor receives the final agent response and yields it as the workflow output.
    """
    display(
        HTML("""
        <div style='padding: 15px; background: #f3e5f5; border-left: 4px solid #9c27b0; border-radius: 4px; margin: 10px 0;'>
            <strong>📤 Display Executor:</strong> Yielding workflow output
        </div>
    """)
    )

    await ctx.yield_output(response.agent_run_response.text)


print("✅ display_result executor created with @executor decorator")

## படி 5: சுற்றுச்சூழல் மாறிலிகளை ஏற்றுக

LLM கிளையன்டை உட்கட்டமைக்கவும். இந்த உதாரணம் பணிபுரிகிறது:
- **Microsoft Foundry** / **Azure OpenAI** (பதில் API — பரிந்துரைக்கப்படுகிறது)
- **Azure OpenAI**
- **OpenAI**


In [ ]:
# Load environment variables
load_dotenv()

# Configure the Microsoft Foundry provider with keyless authentication
provider = FoundryChatClient(
    project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
    model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
    credential=AzureCliCredential(),
)


## படி 6: கட்டமைக்கப்பட்ட வெளியீடுகளுடன் AI முகவர்களை உருவாக்குதல்

நாம் ஒவ்வொன்றும் `AgentExecutor` இல் சுருக்கப்பட்ட **மூன்று சிறப்பு முகவர்களை** உருவாக்குகிறோம்:

1. **availability_agent** - உபகரணத்தை பயன்படுத்தி ஹோட்டல் கிடைக்கும் நிலையை சோதிக்கும்
2. **alternative_agent** - மாற்று நகரங்களை பரிந்துரைக்கும் (இறக்குமதி இல்லாவிட்டால்)
3. **booking_agent** - ஹோட்டல் அறைகள் கிடைக்கும் போது முன்பதிவு செய்ய ஊக்குவிக்கும்

**முக்கிய அம்சங்கள்:**
- `tools=[hotel_booking]` - முகவருக்கு உபகரணத்தை வழங்குகிறது
- `response_format=PydanticModel` - கட்டமைக்கப்பட்ட JSON வெளியீட்டை கட்டாயப்படுத்துகிறது
- `AgentExecutor(..., id="...")` - வேலைநெறி பயன்பாட்டிற்கு முகவரை சுருக்குகிறது


In [ ]:
# Agent 1: Check availability with tool
availability_agent = AgentExecutor(
    provider.as_agent(
        name="availability-agent",
        instructions=(
            "You are a hotel booking assistant that checks room availability. "
            "Use the hotel_booking tool to check if rooms are available at the destination. "
            "Return JSON with fields: destination (string), has_availability (bool), and message (string). "
            "The message should summarize the availability status."
        ),
        tools=[hotel_booking],
        default_options={"response_format": BookingCheckResult},
    ),
    id="availability_agent",
)

# Agent 2: Suggest alternative (when no rooms)
alternative_agent = AgentExecutor(
    provider.as_agent(
        name="alternative-agent",
        instructions=(
            "You are a helpful travel assistant. When a user cannot find hotels in their requested city, "
            "suggest an alternative nearby city that has availability. "
            "Return JSON with fields: alternative_destination (string) and reason (string). "
            "Make your suggestion sound appealing and helpful."
        ),
        default_options={"response_format": AlternativeResult},
    ),
    id="alternative_agent",
)

# Agent 3: Suggest booking (when rooms available)
booking_agent = AgentExecutor(
    provider.as_agent(
        name="booking-agent",
        instructions=(
            "You are a booking assistant. The user has found available hotel rooms. "
            "Encourage them to book by highlighting the destination's appeal. "
            "Return JSON with fields: destination (string), action (string), and message (string). "
            "The action should be 'book_now' and message should be encouraging."
        ),
        default_options={"response_format": BookingConfirmation},
    ),
    id="booking_agent",
)

display(
    HTML("""
    <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
        <strong>✅ Created 3 Agents:</strong>
        <ul style='margin: 10px 0 0 0;'>
            <li><strong>availability_agent</strong> - Checks availability with hotel_booking tool</li>
            <li><strong>alternative_agent</strong> - Suggests alternative cities</li>
            <li><strong>booking_agent</strong> - Encourages booking</li>
        </ul>
    </div>
""")
)


## படி 7: நிபந்தனை கூரைகளுடன் வேலைசூழலை கட்டமைக்கவும்

இப்போது நாங்கள் `WorkflowBuilder` ஐ நிபந்தனை வழிசெலுத்தலுடன் கூடிய வரைவை கட்டமைக்க பயன்படுத்துகிறோம்:

**வேலைசூழல் கட்டமைப்பு:**
```
availability_agent (START)
        ↓
   Evaluate conditions
        ↙         ↘
[no_availability]  [has_availability]
        ↓              ↓
alternative_agent  booking_agent
        ↓              ↓
    display_result ←───┘
```

**முக்கிய முறைகள்:**
- `.set_start_executor(...)` - ஆரம்ப இடத்தை அமைக்கிறது
- `.add_edge(from, to, condition=...)` - நிபந்தனை கூரையை சேர்க்கிறது
- `.build()` - வேலைசூழலை இறுதிப்படுத்துகிறது


In [ ]:
# Build the workflow with conditional routing
workflow = (
    WorkflowBuilder(
        start_executor=availability_agent,
        output_executors=[display_result],
    )
    # NO AVAILABILITY PATH
    .add_edge(availability_agent, alternative_agent, condition=no_availability_condition)
    .add_edge(alternative_agent, display_result)
    # HAS AVAILABILITY PATH
    .add_edge(availability_agent, booking_agent, condition=has_availability_condition)
    .add_edge(booking_agent, display_result)
    .build()
)

display(
    HTML("""
    <div style='padding: 20px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; border-radius: 8px; margin: 10px 0;'>
        <h3 style='margin: 0 0 15px 0;'>✅ Workflow Built Successfully!</h3>
        <p style='margin: 0; line-height: 1.6;'>
            <strong>Conditional Routing:</strong><br>
            • If <strong>NO availability</strong> → alternative_agent → display_result<br>
            • If <strong>availability</strong> → booking_agent → display_result
        </p>
    </div>
""")
)

## படி 8: சோதனை வழக்கு 1 - கிடைக்கும் இடமில்லாத நகரம் (பெரிஸ்)

பெரிசில் உள்ள ஹோட்டல்களை கேட்கும்போது **கிடைக்கும் இடம் இல்லாத** பாதையை சோதிக்க நடத்துவோம் (எங்கள் சிமுலேஷனில் அது எந்த அறைகளும் இல்லாதது).


In [ ]:
display(
    HTML("""
    <div style='padding: 20px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #e65100;'>🧪 TEST CASE 1: Paris (No Availability)</h3>
        <p style='margin: 0;'>Expected workflow path: availability_agent → alternative_agent → display_result</p>
    </div>
""")
)

# Create request for Paris
request_paris = AgentExecutorRequest(
    messages=[Message(role="user", text="I want to book a hotel in Paris")], should_respond=True
)

# Run the workflow
events_paris = await workflow.run(request_paris)
outputs_paris = events_paris.get_outputs()

# Display results
if outputs_paris:
    result_paris = AlternativeResult.model_validate_json(outputs_paris[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); border-radius: 12px; box-shadow: 0 4px 12px rgba(255,165,0,0.3); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0; color: #333;'>🏆 WORKFLOW RESULT (Paris)</h3>
            <div style='background: white; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ❌ No rooms in Paris</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Alternative Suggestion:</strong> 🏨 {result_paris.alternative_destination}</p>
                <p style='margin: 0; font-size: 14px; color: #666;'><strong>Reason:</strong> {result_paris.reason}</p>
            </div>
        </div>
    """)
    )

## படி 9: சோதனை வழக்கு 2 இயக்குக - நகரம் கிடைக்கும் இருப்புடன் (ஸ்டாக்ஹோம்)

இப்போது ஸ்டாக்ஹோம் நகரத்தில் ஹோட்டல்களை கேட்டு **கிடைக்கும்** பாதையைச் சோதிப்போம் (எங்கள் மாதிரியில் அறைகள் உள்ள நகரம்).


In [ ]:
display(
    HTML("""
    <div style='padding: 20px; background: #e8f5e9; border-left: 4px solid #4caf50; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #1b5e20;'>🧪 TEST CASE 2: Stockholm (Has Availability)</h3>
        <p style='margin: 0;'>Expected workflow path: availability_agent → booking_agent → display_result</p>
    </div>
""")
)

# Create request for Stockholm
request_stockholm = AgentExecutorRequest(
    messages=[Message(role="user", text="I want to book a hotel in Stockholm")], should_respond=True
)

# Run the workflow
events_stockholm = await workflow.run(request_stockholm)
outputs_stockholm = events_stockholm.get_outputs()

# Display results
if outputs_stockholm:
    result_stockholm = BookingConfirmation.model_validate_json(outputs_stockholm[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #4caf50 0%, #8bc34a 100%); color: white; border-radius: 12px; box-shadow: 0 4px 12px rgba(76,175,80,0.3); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0;'>🏆 WORKFLOW RESULT (Stockholm)</h3>
            <div style='background: white; color: #333; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ✅ Rooms Available!</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Destination:</strong> 🏨 {result_stockholm.destination}</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Action:</strong> {result_stockholm.action}</p>
                <p style='margin: 0; font-size: 14px; color: #666;'><strong>Message:</strong> {result_stockholm.message}</p>
            </div>
        </div>
    """)
    )

## முக்கியமான எடுத்துக்காட்டுகள் மற்றும் அடுத்த படிகள்

### ✅ நீங்கள் கற்றுக்கொண்டது:

1. **WorkflowBuilder படிவம்**
   - நுழைவு புள்ளியை வரையறுக்க `.set_start_executor()` ஐ பயன்படுத்தவும்
   - நிபந்தனை உள்ள வழிமுறைக்கு `.add_edge(from, to, condition=...)` ஐ பயன்படுத்தவும்
   - workflow ஐ இறுதிப்படுத்த `.build()` ஐ அழைக்கவும்

2. **நிபந்தனை வழித்தடம்**
   - நிலைமை செயல்பாடுகள் `AgentExecutorResponse` ஐ பரிசீலிக்கின்றன
   - வழிப்பெறல் முடிவுகளை எடுக்க கட்டமைக்கப்பட்ட வெளியீடுகளை பகுப்பாய்வு செய்கின்றன
   - ஒரு எதிர் செயல்படுத்த `True` ஐ, தாண்ட `False` ஐ திருப்பவும்

3. **கருவி ஒருங்கிணைப்பு**
   - Python செயல்பாடுகளை AI கருவிகளாக மாற்ற `@ai_function` ஐ பயன்படுத்தவும்
   - தேவையானபோது முகவர்கள் கருவிகளை தானாகக் கூப்பிடுகின்றனர்
   - கருவிகள் முகவர்கள் பகுப்பாய்வு செய்யக்கூடிய JSON ஐ திருப்புகின்றன

4. **கட்டமைக்கப்பட்ட வெளியீடுகள்**
   - விண்ணப்பம் பாதுகாப்பானத் தரவு எடுக்க Pydantic மாதிரிகளை பயன்படுத்தவும்
   - முகவர்களை உருவாக்கும்போது `response_format=MyModel` ஐ அமைக்கவும்
   - `Model.model_validate_json()` உடன் பதில்களை பகுப்பாய்வு செய்யவும்

5. **தனிப்பயன் நிருவாகிகள்**
   - workflow கூறுகளை உருவாக்க `@executor(id="...")` பயன்படுத்தவும்
   - நிருவாகிகள் தரவை மாற்றவோ அல்லது பக்க விளைவுகளைச் செய்யவோ முடியும்
   - workflow முடிவுகளை உருவாக்க `ctx.yield_output()` ஐ பயன்படுத்தவும்

### 🚀 நிஜ உலக பயன்பாடுகள்:

- **பயண முன்பதிவு**: கிடைக்கும் நிலையை சரிபார், மாற்று பரிந்துரைகள், விருப்பங்களை ஒப்பீடு செய்க
- **வாடிக்கையாளர் சேவை**: பிரச்சினை வகை, மனோபாவம், முன்னுரிமை அடிப்படையில் வழிமுறை அமைக்கவும்
- **மின்னணு வர்த்தகம்**: சரக்குகளைச் சரிபார், மாற்றுகளை பரிந்துரைக்கவும், ஆர்டர்களை செயலாக்கவும்
- **உள்ளடக்க ஒழுக்காமை**: நச்சுத்தன்மை மதிப்பெண்கள், பயனர் கொடிகளின் அடிப்படையில் வழிமுறைகள்
- **அங்கீகார workflow கள்**: தொகை, பயனர் பங்கு, ஆபத்து நிலை அடிப்படையில் வழிமுறை அமைக்கவும்
- **பல படி செயலாக்கம்**: தர தரம், முழுமை அடிப்படையில் வழிமுறை அமைக்கவும்

### 📚 அடுத்த படிகள்:

- மேலும் சிக்கலான நிபந்தனைகள் (பல வழிகாட்டிகள்) சேர்
- workflow நிலை மேலாண்மையுடன் முறைகளை செயல்படுத்து
- மீண்டும் பயன்படுத்தக்கூடிய கூறுகளுக்கான துணை workflow களைச் சேர்க்கவும்
- உண்மையான API களுடன் ஒருங்கிணைத்தல் (ஹோட்டல் முன்பதிவு, சரக்கு அமைப்புகள்)
- பிழை கையாளுதல் மற்றும் மாற்று வழிமுறைகள் சேர்க்கவும்
- உள்ளமைவு காட்சிப்படுத்தல் கருவிகளுடன் workflow களை காட்சிப்படுத்தவும்


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**மறுப்பு**:
இந்த ஆவணம் AI மொழிபெயர்ப்பு சேவை [Co-op Translator](https://github.com/Azure/co-op-translator) பயன்படுத்தி மொழிபெயர்க்கப்பட்டுள்ளது. நாங்கள் துல்லியத்திற்காக முயற்சி செய்துள்ளோம், ஆனால் தானாக செய்யப்படும் மொழிபெயர்ப்புகளில் பிழைகள் அல்லது தவறுகள் இருக்கலாம் என்பதை கவனத்தில் கொள்ளவும். அசல் ஆவணம் அதன் தாய்மொழியில் அதிகாரப்பூர்வ ஆதாரமாக கருதப்பட வேண்டும். முக்கியமான தகவல்களுக்கு, தொழில்நுட்பமான மனித மொழிபெயர்ப்பு பரிந்துரைக்கப்படுகிறது. இந்த மொழிபெயர்ப்பைப் பயன்படுத்துவதால் ஏற்படும் எந்த தவறான புரிதல்கள் அல்லது தவறான விளக்கத்திற்கும் நாங்கள் பொறுப்பில்வில்லை.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
